In [ ]:
import torch
import pandas as pd
import os
from tqdm.auto import tqdm
from google.colab import drive
drive.mount("/content/drive")

import torch.nn as nn
import torch.nn.functional as F

class SAE(nn.Module):
    def __init__(self, d_in, d_hidden, l1_coeff=3e-4):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_hidden, bias=False)
        self.decoder = nn.Linear(d_hidden, d_in, bias=False)
        self.l1_coeff = l1_coeff

        # Tied weights option (recommended for interpretability)
        self.decoder.weight = nn.Parameter(self.encoder.weight.T)

    def forward(self, x):
        # Sparse latent features
        z = F.relu(self.encoder(x))
        # Reconstruction
        x_hat = self.decoder(z)
        return x_hat, z

    def loss(self, x, x_hat, z):
        mse = ((x - x_hat)**2).mean()
        l1 = self.l1_coeff * z.abs().sum(dim=-1).mean()
        return mse + l1

# ------------------------------------
# Paths
# ------------------------------------
BASE_DIR = "/content/drive/MyDrive/SAE_Works_Pythia-2.8B"
CSV_PATH = "/content/drive/MyDrive/Pythia2.8B_reward_hacking_results.csv"
OUT_CSV = "/content/drive/MyDrive/SAE_Works/sae_neuron_stats.csv"

# ------------------------------------
# Load CSV
# ------------------------------------
df = pd.read_csv(CSV_PATH)
saes = []
for i in range(4):
  # Modified path construction to use os.path.join for robustness
  sae_path = os.path.join(BASE_DIR, f"layer 24/sae_mixed{i+1}.pt")
  # FIX: Add weights_only=False to torch.load to allow loading of custom classes
  sae = torch.load(sae_path, map_location="cuda", weights_only=False)
  sae.eval()
  sae.cuda()
  # Fix: The variable was `loaded_sae` but should be `sae`
  saes.append(sae)
# Ensure boolean
df["reward_hacked"] = df["reward_hacked"].astype(bool)

# ------------------------------------
# Helper: load activations
# ------------------------------------
def load_activations(run_id):
    path = os.path.join(BASE_DIR, f"run_{run_id:05d}.pt")
    return torch.load(path)  # [1, seq, d_model]

# ------------------------------------
# Main loop
# ------------------------------------
rows = []

for sae_idx, sae in enumerate(saes):  # saes = trained Matryoshka SAEs
    print(f"\nProcessing SAE {sae_idx}")
    sae.eval().cuda()

    hacked_z = []
    clean_z = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        run_id = int(row["run_id"])
        hacked = row["reward_hacked"]
        sample = torch.load(f"/content/drive/MyDrive/SAE_Works_Pythia-2.8B/run_{run_id:05d}.pt")

    # sample["activations"] is a list of [1, seq, d_model] tensors
        acts_list = sample["activations"]

    # Concatenate across generation steps
        acts = torch.cat(acts_list, dim=1)          # [1, seq, d_model]
        acts = acts.reshape(-1, acts.shape[-1])  # [tokens, d_model]
        acts = acts.to(torch.float32) # Convert to float32 to match SAE weights
        acts = acts.cuda()


        with torch.no_grad():
            _, z = sae(acts)  # [tokens, d_hidden]

        z = z.cpu()

        if hacked:
            hacked_z.append(z)
        else:
            clean_z.append(z)

    # Concatenate
    hacked_z = torch.cat(hacked_z, dim=0)
    clean_z = torch.cat(clean_z, dim=0)

    # ------------------------------------
    # Compute statistics
    # ------------------------------------
    freq_h = (hacked_z > 0).float().mean(dim=0)
    freq_c = (clean_z > 0).float().mean(dim=0)

    mean_h = hacked_z.mean(dim=0)
    mean_c = clean_z.mean(dim=0)

    delta = freq_h - freq_c
    selectivity = mean_h / (mean_c + 1e-6)

    # ------------------------------------
    # Save per-neuron
# ------------------------------------
    for i in range(sae.encoder.out_features):
        rows.append({
            "sae_idx": sae_idx,
            "neuron_id": i,
            "freq_hacked": freq_h[i].item(),
            "freq_clean": freq_c[i].item(),
            "delta_freq": delta[i].item(),
            "mean_hacked": mean_h[i].item(),
            "mean_clean": mean_c[i].item(),
            "selectivity": selectivity[i].item(),
        })

# ------------------------------------
# Save CSV
# ------------------------------------
stats_df = pd.DataFrame(rows)
stats_df.to_csv(OUT_CSV, index=False)

print(f"✓ Saved neuron statistics to {OUT_CSV}")

Mounted at /content/drive

Processing SAE 0


  0%|          | 0/2985 [00:00<?, ?it/s]

KeyboardInterrupt: 

Let's first verify that a GPU is indeed connected and recognized by the system.

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA device found. Please ensure your Colab runtime type is set to GPU (Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU).")

# Also check with nvidia-smi for system-level GPU info
!nvidia-smi

In [ ]:
import torch
import pandas as pd

# ----------------------------
# Inputs (already computed)
# ----------------------------
# hacked_z_list: List[Tensor(tokens, d_hidden)]
# clean_z_list:  List[Tensor(tokens, d_hidden)]

# NOTE: hacked_z and clean_z are already concatenated tensors from the previous cell.
# The comments above about 'hacked_z_list' are misleading for the current state.

assert len(hacked_z) > 0, "No hacked activations found"
assert len(clean_z) > 0, "No clean activations found"

# ----------------------------
# Stack all tokens (hacked_z and clean_z are already stacked from the previous cell)
# ----------------------------
# hacked_z_stack = torch.cat(hacked_z, dim=0)   # [N_hacked_tokens, d_hidden]
# clean_z_stack  = torch.cat(clean_z, dim=0)    # [N_clean_tokens, d_hidden]

d_hidden = hacked_z.shape[1]

# ----------------------------
# Core statistics
# ----------------------------
mean_hacked = hacked_z.mean(dim=0)
mean_clean  = clean_z.mean(dim=0)

var_hacked  = hacked_z.var(dim=0)

freq_hacked = (hacked_z > 0).float().mean(dim=0)
freq_clean  = (clean_z > 0).float().mean(dim=0)

delta       = mean_hacked - mean_clean
ratio       = mean_hacked / (mean_clean + 1e-6)
freq_diff  = freq_hacked - freq_clean

# ----------------------------
# Composite causal score
# ----------------------------
score = (delta * freq_diff) / (var_hacked.sqrt() + 1e-6)

# ----------------------------
# Build DataFrame
# ----------------------------
df = pd.DataFrame({
    "neuron_id": range(d_hidden),
    "mean_hacked": mean_hacked.cpu().numpy(),
    "mean_clean": mean_clean.cpu().numpy(),
    "freq_hacked": freq_hacked.cpu().numpy(),
    "freq_clean": freq_clean.cpu().numpy(),
    "variance_hacked": var_hacked.cpu().numpy(),
    "delta": delta.cpu().numpy(),
    "ratio": ratio.cpu().numpy(),
    "freq_diff": freq_diff.cpu().numpy(),
    "score": score.cpu().numpy(),
})

# ----------------------------
# Filter junk neurons
# ----------------------------
df = df[
    (df["freq_hacked"] >= 0.05) &    # fires at least 5% of hacked tokens
    (df["delta"] > 0)                # higher on hacked than clean
]

# ----------------------------
# Rank neurons
# ----------------------------
df = df.sort_values("score", ascending=False)

# ----------------------------
# Save results
# ----------------------------
csv_path = "reward_hacking_neuron_ranking_pythia-2.8B_layer_24.csv"
df.to_csv(csv_path, index=False)

print(f"✓ Saved neuron ranking to {csv_path}")
print("\nTop 10 candidate reward-hacking neurons:")
display(df.head(10))

In [ ]:
import os
import shutil

source_path = "reward_hacking_neuron_ranking.csv"
destination_folder = "/content/drive/MyDrive/SAE_Works_Pythia-2.8B"
destination_path = os.path.join(destination_folder, source_path)

# Ensure the destination directory exists
os.makedirs(destination_folder, exist_ok=True)

# Use shutil.move to handle cross-device links (copy then delete)
shutil.move(source_path, destination_path)

print(f"✓ Moved '{source_path}' to '{destination_path}'")

FileNotFoundError: [Errno 2] No such file or directory: 'reward_hacking_neuron_ranking.csv'

In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/SAE_Works/Pythia-2.8B"
CSV_PATH = "/content/drive/MyDrive/Pythia2.8B_reward_hacking_results.csv"

df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print(df["reward_hacked"].value_counts())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 2985
reward_hacked
False    1580
True     1405
Name: count, dtype: int64


In [ ]:
hacked_acts = []
clean_acts  = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    run_id = int(row["run_id"])
    hacked = int(row["reward_hacked"]) == 1  # ✅ FIX

    path = f"/content/drive/MyDrive/SAE_Works_Pythia-2.8B/run_{run_id:05d}.pt"
    if not os.path.exists(path):
        continue

    sample = torch.load(path)

    acts_list = sample["activations"]  # list of [1, seq, d_model]

    acts = torch.cat(acts_list, dim=1)       # [1, seq, d_model]
    acts = acts.reshape(-1, acts.shape[-1])  # [tokens, d_model]
    acts = acts.float()                      # fp32 for SAE

    if hacked:
        hacked_acts.append(acts)
    else:
        clean_acts.append(acts)

X_hacked = torch.cat(hacked_acts, dim=0)
X_clean  = torch.cat(clean_acts, dim=0)

print("Hacked tokens:", X_hacked.shape)
print("Clean tokens :", X_clean.shape)


100%|██████████| 2985/2985 [25:43<00:00,  1.93it/s]


Hacked tokens: torch.Size([177298, 2560])
Clean tokens : torch.Size([193267, 2560])


In [ ]:
def rank_sae_neurons(
    sae,
    X_hacked,
    X_clean,
    sae_level,
    min_freq=0.05,
):
    sae.eval()
    sae.cuda()

    with torch.no_grad():
        Z_h = torch.relu(sae.encoder(X_hacked.cuda())).cpu()
        Z_c = torch.relu(sae.encoder(X_clean.cuda())).cpu()

    mean_h = Z_h.mean(dim=0)
    mean_c = Z_c.mean(dim=0)

    var_h  = Z_h.var(dim=0)

    freq_h = (Z_h > 0).float().mean(dim=0)
    freq_c = (Z_c > 0).float().mean(dim=0)

    delta     = mean_h - mean_c
    freq_diff = freq_h - freq_c

    score = (delta * freq_diff) / (var_h.sqrt() + 1e-6)

    df = pd.DataFrame({
        "sae_level": sae_level,
        "neuron_id": range(Z_h.shape[1]),
        "mean_hacked": mean_h.numpy(),
        "mean_clean": mean_c.numpy(),
        "freq_hacked": freq_h.numpy(),
        "freq_clean": freq_c.numpy(),
        "variance_hacked": var_h.numpy(),
        "delta": delta.numpy(),
        "freq_diff": freq_diff.numpy(),
        "score": score.numpy(),
    })

    # filter junk
    df = df[
        (df["freq_hacked"] >= min_freq) &
        (df["delta"] > 0)
    ]

    df = df.sort_values("score", ascending=False)

    return df


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/SAE_Works_Pythia-2.8B/layer 24/reward_hacking_neuron_ranking_pythia-2.8B_layer_24.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_rankings = []

for i, sae in enumerate(saes):
    print(f"\nRanking neurons for SAE {i}...")
    df_rank = rank_sae_neurons(
        sae,
        X_hacked,
        X_clean,
        sae_level=i
    )

    path = os.path.join(OUTPUT_DIR, f"sae_{i}_neuron_ranking.csv")
    df_rank.to_csv(path, index=False)

    print(f"Saved → {path}")
    display(df_rank.head(10))

    all_rankings.append(df_rank)



Ranking neurons for SAE 0...
Saved → /content/drive/MyDrive/SAE_Works_Pythia-2.8B/layer 24/reward_hacking_neuron_ranking_pythia-2.8B_layer_24.csv/sae_0_neuron_ranking.csv


,sae_level,neuron_id,mean_hacked,mean_clean,freq_hacked,freq_clean,variance_hacked,delta,freq_diff,score
1550,0,1550,0.070602,0.059854,0.402091,0.379811,0.034416,0.010748,0.022280,0.001291
1213,0,1213,0.066688,0.058244,0.364652,0.343375,0.025642,0.008444,0.021277,0.001122
1927,0,1927,0.088503,0.082338,0.584795,0.566972,0.017371,0.006165,0.017823,0.000834
1265,0,1265,0.024522,0.015387,0.057592,0.046200,0.026974,0.009134,0.011392,0.000634
1998,0,1998,0.068706,0.063376,0.438730,0.425644,0.018393,0.005329,0.013086,0.000514
708,0,708,0.065275,0.059745,0.422904,0.412569,0.017414,0.005531,0.010335,0.000433
53,0,53,0.051423,0.047827,0.354640,0.341331,0.012327,0.003596,0.013309,0.000431
1068,0,1068,0.071024,0.068004,0.493463,0.477624,0.012816,0.003020,0.015839,0.000423
1172,0,1172,0.035499,0.032386,0.233618,0.221181,0.008552,0.003113,0.012437,0.000419
546,0,546,0.045698,0.042980,0.332649,0.321058,0.009675,0.002717,0.011591,0.000320



Ranking neurons for SAE 1...
Saved → /content/drive/MyDrive/SAE_Works_Pythia-2.8B/layer 24/reward_hacking_neuron_ranking_pythia-2.8B_layer_24.csv/sae_1_neuron_ranking.csv


,sae_level,neuron_id,mean_hacked,mean_clean,freq_hacked,freq_clean,variance_hacked,delta,freq_diff,score
1121,1,1121,0.119969,0.112460,0.540993,0.521237,0.027930,0.007509,0.019756,0.000888
276,1,276,0.108175,0.103628,0.598371,0.577941,0.018843,0.004547,0.020430,0.000677
556,1,556,0.117293,0.111563,0.557869,0.540191,0.024802,0.005730,0.017678,0.000643
665,1,665,0.099440,0.095520,0.575449,0.555056,0.017362,0.003920,0.020393,0.000607
1912,1,1912,0.056708,0.053570,0.345915,0.326817,0.013339,0.003138,0.019097,0.000519
1363,1,1363,0.056192,0.053115,0.349716,0.331562,0.012255,0.003077,0.018154,0.000505
1012,1,1012,0.126494,0.121395,0.563233,0.547057,0.029028,0.005100,0.016176,0.000484
2047,1,2047,0.041806,0.039092,0.310145,0.294742,0.007974,0.002715,0.015402,0.000468
970,1,970,0.191678,0.185813,0.670837,0.654023,0.047462,0.005865,0.016814,0.000453
1339,1,1339,0.117445,0.112798,0.551501,0.536532,0.026523,0.004647,0.014969,0.000427



Ranking neurons for SAE 2...
Saved → /content/drive/MyDrive/SAE_Works_Pythia-2.8B/layer 24/reward_hacking_neuron_ranking_pythia-2.8B_layer_24.csv/sae_2_neuron_ranking.csv


,sae_level,neuron_id,mean_hacked,mean_clean,freq_hacked,freq_clean,variance_hacked,delta,freq_diff,score
1452,2,1452,0.486554,0.461747,0.557654,0.539885,0.438314,0.024808,0.017769,0.000666
1685,2,1685,0.422992,0.397047,0.386897,0.367476,0.575469,0.025945,0.019421,0.000664
1215,2,1215,0.249701,0.233523,0.392808,0.374999,0.203505,0.016178,0.017808,0.000639
1003,2,1003,0.477442,0.446925,0.437309,0.421158,0.608261,0.030517,0.016151,0.000632
1464,2,1464,0.522351,0.498712,0.539826,0.522246,0.530211,0.023640,0.017579,0.000571
1089,2,1089,0.649433,0.617079,0.560192,0.544790,0.783968,0.032354,0.015402,0.000563
969,2,969,0.349724,0.327307,0.416519,0.401543,0.364557,0.022417,0.014976,0.000556
153,2,153,0.753232,0.722980,0.655817,0.639773,0.764765,0.030252,0.016044,0.000555
715,2,715,0.323464,0.300125,0.341431,0.326341,0.414205,0.023339,0.015090,0.000547
1822,2,1822,0.675918,0.650307,0.576639,0.558000,0.770498,0.025611,0.018639,0.000544



Ranking neurons for SAE 3...
Saved → /content/drive/MyDrive/SAE_Works_Pythia-2.8B/layer 24/reward_hacking_neuron_ranking_pythia-2.8B_layer_24.csv/sae_3_neuron_ranking.csv


,sae_level,neuron_id,mean_hacked,mean_clean,freq_hacked,freq_clean,variance_hacked,delta,freq_diff,score
55,3,55,0.439030,0.423233,0.922137,0.903781,0.080422,0.015797,0.018356,0.001023
1665,3,1665,0.107750,0.101994,0.462058,0.443718,0.030250,0.005755,0.018340,0.000607
1823,3,1823,0.208703,0.198197,0.543847,0.528145,0.081619,0.010506,0.015702,0.000577
1461,3,1461,0.448232,0.433139,0.808424,0.793581,0.156891,0.015094,0.014843,0.000566
1250,3,1250,0.234219,0.224106,0.585698,0.569326,0.090097,0.010113,0.016371,0.000552
1449,3,1449,0.821088,0.794447,0.859683,0.847253,0.404450,0.026640,0.012430,0.000521
779,3,779,0.331699,0.321621,0.825728,0.811965,0.072827,0.010077,0.013764,0.000514
1558,3,1558,0.598966,0.583836,0.830489,0.815080,0.273663,0.015130,0.015409,0.000446
234,3,234,0.570804,0.558223,0.914167,0.901571,0.136519,0.012582,0.012596,0.000429
1864,3,1864,0.630205,0.611520,0.800065,0.787408,0.321223,0.018686,0.012657,0.000417
